In [1]:
%load_ext autoreload
%autoreload 2 

In [2]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), os.pardir)))
import scripts.extract_features as extract_features

In [3]:
import pandas as pd

In [4]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

In [5]:
df=pd.read_json("./data/raw/dataset.json")

In [6]:
def preview(df):
    print("Dataframe shape:", df.shape)
    print("\nDataframe info:")
    print(df.info())
    print("\nDataframe columns:", df.columns)
    print("\nDataframe unique values count:")
    for colmn in df.columns:
        print(f"{colmn}: {df[colmn].nunique()}")

In [7]:

preview(df)

Dataframe shape: (28550, 6)

Dataframe info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28550 entries, 0 to 28549
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   brand    28550 non-null  object 
 1   name     28550 non-null  object 
 2   price    28550 non-null  object 
 3   rating   28550 non-null  float64
 4   reviews  28550 non-null  int64  
 5   link     28550 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 1.3+ MB
None

Dataframe columns: Index(['brand', 'name', 'price', 'rating', 'reviews', 'link'], dtype='object')

Dataframe unique values count:
brand: 10
name: 25542
price: 14326
rating: 25
reviews: 101
link: 25584


In [8]:
df["price"]=df["price"].str.replace("TL","").str.replace(".","").str.replace(",",".").astype(float)
df["rating"]=df["rating"].astype(float)
df["reviews"]=df["reviews"].astype(int)
df["link"]=df["link"].astype(object)
df["brand"]=df["brand"].astype(object)

In [9]:
df.drop(columns=["reviews","link","rating"], inplace=True)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28550 entries, 0 to 28549
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   brand   28550 non-null  object 
 1   name    28550 non-null  object 
 2   price   28550 non-null  float64
dtypes: float64(1), object(2)
memory usage: 669.3+ KB


In [11]:
df.name.head()

0    TUF A16 FA607NUG-RL125-Gaming AMD Ryzen 7 7445HS 16GB 512GB SSD RTX 4050 8GB 16" FHD+ WUXGA IPS Fdos
1                                 Vivobook 15 X1504VA Intel Core 5 120U 16GB 512GB SSD 15.6" W11 + OFFİCE
2                 Vivobook Go Intel Celeron N4500 4GB 128GB SSD Windows 11 Home 15.6'' FHD E510KA-EJ1253W
3    TUF Gaming A16 FA607NUG-RL212 R7 7445HS 16GB 512GB SSD RTX4050-6GB 140W Dos 16" WUXGA 144Hz Notebook
4      TUF F16 FX608JH-RV010-Gaming Intel Core i5-13450HX 16GB 512GB SSD RTX 5050 16" FHD+ WUXGA IPS Fdos
Name: name, dtype: object

In [12]:
## ASUS LAPTOP
asus_laptop=df[df["brand"]=="ASUS"].copy()

In [13]:
asus_laptop["name"].__len__()

4561

In [14]:
def extract_features_from_name(name):
    cpu=extract_features.extract_cpu(name)
    gpu=extract_features.extract_gpu(name)
    ram=extract_features.extract_ram(name)
    storage=extract_features.extract_storage(name)
    os=extract_features.extract_os(name)
    return cpu, gpu, ram, storage

asus_laptop["cpu"]=asus_laptop["name"].apply(lambda x: extract_features.extract_cpu(x))
asus_laptop["ram"]=asus_laptop["name"].apply(lambda x: extract_features.extract_ram(x))
asus_laptop["gpu"]=asus_laptop["name"].apply(lambda x: extract_features.extract_gpu(x))
asus_laptop["storage"]=asus_laptop["name"].apply(lambda x: extract_features.extract_storage(x))
asus_laptop["os"]=asus_laptop["name"].apply(lambda x: extract_features.extract_os(x))


ValueError: invalid literal for int() with base 10: ''

In [ ]:
asus_laptop.head()

,brand,name,price,cpu,ram,gpu,storage,os
0,ASUS,"TUF A16 FA607NUG-RL125-Gaming AMD Ryzen 7 7445HS 16GB 512GB SSD RTX 4050 8GB 16"" FHD+ WUXGA IPS Fdos",43146.0,Amd Ryzen 7 7445Hs,16 GB,RTX 4050 8GB,512 GB SSD,Fdos
1,ASUS,"Vivobook 15 X1504VA Intel Core 5 120U 16GB 512GB SSD 15.6"" W11 + OFFİCE",29149.0,Intel Core 5 120U,16 GB,None,512 GB SSD,W11
2,ASUS,Vivobook Go Intel Celeron N4500 4GB 128GB SSD Windows 11 Home 15.6'' FHD E510KA-EJ1253W,9999.0,Intel Celeron N4500,4 GB,None,128 GB SSD,Windows 11
3,ASUS,"TUF Gaming A16 FA607NUG-RL212 R7 7445HS 16GB 512GB SSD RTX4050-6GB 140W Dos 16"" WUXGA 144Hz Notebook",42449.0,Amd R7 7445Hs,16 GB,RTX4050 6GB,512 GB SSD,Dos
4,ASUS,"TUF F16 FX608JH-RV010-Gaming Intel Core i5-13450HX 16GB 512GB SSD RTX 5050 16"" FHD+ WUXGA IPS Fdos",48799.1,Intel Core Intel Core I5 13450Hx,16 GB,RTX 5050,512 GB SSD,Fdos


In [ ]:
import tests.testFeatures as testFeatures

testFeatures.test_all_with_summary(asus_laptop["name"])


TEST SUMMARY
Total test cases: 4561
CPU found: 4559 (100.0%)
GPU found: 2402 (52.7%)
RAM found: 4547 (99.7%)
Storage found: 4557 (99.9%)
OS found: 4490 (98.4%)


In [ ]:
asus_laptop.cpu.unique()

array(['Amd Ryzen 7 7445Hs', 'Intel Core 5 120U', 'Intel Celeron N4500',
       'Amd R7 7445Hs', 'Intel Core Intel Core I5 13450Hx',
       'Amd Ryzen 5 7520U', 'I5 13450Hx', '1920', 'Intel Celeron N4020',
       'Ryzen 7', 'Amd Ryzen5 7520U', 'Core 5 210H',
       'Intel Core Intel Core I7 14650Hx', 'Intel I7 14650Hx',
       'Ryzen 9 9955Hx', 'Amd Ryzen 5 7535Hs', 'Snapdragon X1',
       'Intel® Core™ 5 120U', 'Intel Core I5 1135G7',
       'Intel Core I5 1334U', 'Intel Core I5 13450Hx', 'Amd Amd R5',
       'Amd Ryzen 9 8940Hx', 'Intel Core I7 14650Hx', 'Amd Amd R9',
       'Intel Core Intel Core I5 13420H', 'İntel Core İ5', 'M1',
       'Ryzen 7 7445Hs', 'Intel Core I7 13620H',
       'Intel Core Intel I5 13450Hx', 'Intel Core Intel Core I7 13620H',
       'I5 13420H', 'Core 5', 'Snapdragon 16Gb',
       'Intel Core Core I7 13620H', 'Intel Core 5 120', 'Core5 210H',
       'Ryzen 9', 'Ultra 9 275Hx', 'Intel Core 5 210H', 'Amd R7',
       'Amd Ryzen 7 5825U', 'Ultra 7 255H', 'Intel 